# N06 – Resultados y selección de modelo (nivel USD/UYU)
Este notebook consolida resultados **Benchmarks + Machine Learning + Deep Learning**, arma un set unificado de predicciones OOS en **nivel (UYU por USD)**, y exporta `best_models_per_horizon` para el notebook de producción.

## 0) Imports y configuración
Definimos rutas, archivos esperados y carpeta de salida.

In [1]:

import os
import re
import json
import glob
from ast import literal_eval

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

# -------------------------
# Root del proyecto
# -------------------------
try:
    ROOT_DIR = os.path.dirname(os.path.abspath(__file__))  # si corrés como .py
except NameError:
    ROOT_DIR = os.getcwd()  # si corrés en notebook

# -------------------------
# Carpetas de resultados (ajustar si cambian)
# -------------------------
RESULTS_BENCH_DIR = os.path.join(ROOT_DIR, "results_n03")
RESULTS_ML_DIR    = os.path.join(ROOT_DIR, "results_n04")
RESULTS_DL_DIR    = os.path.join(ROOT_DIR, "results_n05")

# -------------------------
# Archivos de resultados (csv o xlsx)
# -------------------------
BENCH_RESULTS_BASENAMES = ["fx_results_benchmarks.csv", "fx_results_benchmarks.xlsx"]
ML_RESULTS_BASENAMES    = ["fx_results_ml.csv", "fx_results_ml.xlsx"]
DL_RESULTS_BASENAMES    = ["fx_results_dl.csv", "fx_results_dl.xlsx"]

# -------------------------
# Carpetas OOS
# -------------------------
OOS_DIRS = [
    os.path.join(RESULTS_BENCH_DIR, "oos"),
    os.path.join(RESULTS_ML_DIR, "oos"),
    os.path.join(RESULTS_DL_DIR, "oos"),
]

# -------------------------
# Salidas N06
# -------------------------
OUT_DIR = os.path.join(ROOT_DIR, "results_n06")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_RESULTS_ALL_CSV   = os.path.join(OUT_DIR, "fx_results_all.csv")
OUT_RESULTS_ALL_XLSX  = os.path.join(OUT_DIR, "fx_results_all.xlsx")
OUT_BEST_CSV          = os.path.join(OUT_DIR, "best_models_per_horizon.csv")
OUT_BEST_XLSX         = os.path.join(OUT_DIR, "best_models_per_horizon.xlsx")
OUT_OOS_ALL_CSV       = os.path.join(OUT_DIR, "all_oos_predictions.csv")
OUT_OOS_ALL_XLSX      = os.path.join(OUT_DIR, "all_oos_predictions.xlsx")

print("ROOT_DIR:", ROOT_DIR)
print("OUT_DIR :", OUT_DIR)
print("OOS_DIRS:", OOS_DIRS)


ROOT_DIR: c:\Users\jbrio\OneDrive - Universidad de Montevideo\Tesis MCD\Flujo de trabajo modelado y predicciones\Flujo sin BEVSA
OUT_DIR : c:\Users\jbrio\OneDrive - Universidad de Montevideo\Tesis MCD\Flujo de trabajo modelado y predicciones\Flujo sin BEVSA\results_n06
OOS_DIRS: ['c:\\Users\\jbrio\\OneDrive - Universidad de Montevideo\\Tesis MCD\\Flujo de trabajo modelado y predicciones\\Flujo sin BEVSA\\results_n03\\oos', 'c:\\Users\\jbrio\\OneDrive - Universidad de Montevideo\\Tesis MCD\\Flujo de trabajo modelado y predicciones\\Flujo sin BEVSA\\results_n04\\oos', 'c:\\Users\\jbrio\\OneDrive - Universidad de Montevideo\\Tesis MCD\\Flujo de trabajo modelado y predicciones\\Flujo sin BEVSA\\results_n05\\oos']


## 1) Funciones auxiliares
Lectura robusta (CSV/XLSX), normalización de columnas y parseo de parámetros.

In [2]:

def _first_existing_file(base_dir: str, candidates: list[str]) -> str | None:
    for name in candidates:
        path = os.path.join(base_dir, name)
        if os.path.isfile(path):
            return path
    return None

def read_table_smart(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if ext == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Extensión no soportada: {ext} | {path}")

def safe_to_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def parse_params_to_json(params):
    """Devuelve string JSON válido (o None)."""
    if params is None or (isinstance(params, float) and np.isnan(params)):
        return None

    # ya es dict
    if isinstance(params, dict):
        try:
            return json.dumps(params, ensure_ascii=False, sort_keys=True)
        except Exception:
            return None

    # string que parece dict/lista
    if isinstance(params, str):
        s = params.strip()
        if not s:
            return None
        # intenta JSON directo
        try:
            obj = json.loads(s)
            return json.dumps(obj, ensure_ascii=False, sort_keys=True)
        except Exception:
            pass
        # intenta python literal (muchos notebooks guardan dict con comillas simples)
        try:
            obj = literal_eval(s)
            if isinstance(obj, (dict, list)):
                return json.dumps(obj, ensure_ascii=False, sort_keys=True)
        except Exception:
            return None

    return None

def ensure_cols(df: pd.DataFrame, required: list[str], where: str):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas en {where}: {missing}\nColumnas disponibles: {list(df.columns)}")
    return df


def sanitize_model_name(name: str) -> str:
    """Normaliza nombre de modelo para matching con nombres de archivos OOS.
    - remueve paréntesis
    - remueve espacios
    - deja solo [A-Za-z0-9_\-] (lo demás lo borra)
    """
    if name is None:
        return ""
    s = str(name)
    s = s.replace("(", "").replace(")", "").replace(" ", "")
    s = re.sub(r"[^A-Za-z0-9_\-]", "", s)
    return s


<>:66: SyntaxWarning: invalid escape sequence '\-'
<>:66: SyntaxWarning: invalid escape sequence '\-'
C:\Users\jbrio\AppData\Local\Temp\ipykernel_12636\1425761976.py:66: SyntaxWarning: invalid escape sequence '\-'
  - deja solo [A-Za-z0-9_\-] (lo demás lo borra)


## 2) Cargar resultados agregados (Bench / ML / DL)
Leemos los archivos `fx_results_*` y armamos un DataFrame único para comparar en **nivel**.

In [3]:
bench_path = _first_existing_file(RESULTS_BENCH_DIR, BENCH_RESULTS_BASENAMES)
ml_path    = _first_existing_file(RESULTS_ML_DIR,    ML_RESULTS_BASENAMES)
dl_path    = _first_existing_file(RESULTS_DL_DIR,    DL_RESULTS_BASENAMES)

print("BENCH:", bench_path)
print("ML   :", ml_path)
print("DL   :", dl_path)

# ------------------------------------------------------------
# Cargamos SOLO las columnas necesarias (según tu estructura)
# ------------------------------------------------------------
NEEDED_RESULTS_COLS = ["h_tag", "H_days", "model", "MAE_level", "RMSE_level", "R2_level", "params"]

dfs = []

def _load_results(path: str, family: str) -> pd.DataFrame:
    df = read_table_smart(path).copy()

    # Selección explícita: solo columnas presentes en el archivo
    keep = [c for c in NEEDED_RESULTS_COLS if c in df.columns]
    df = df[keep].copy()

    # Checks mínimos
    ensure_cols(df, ["h_tag", "model"], f"results ({family})")

    df["family"] = family
    return df

if bench_path:
    dfs.append(_load_results(bench_path, "benchmark"))

if ml_path:
    dfs.append(_load_results(ml_path, "ml"))

if dl_path:
    dfs.append(_load_results(dl_path, "dl"))

if not dfs:
    raise FileNotFoundError("No encontré archivos de resultados (bench/ml/dl). Revisá RESULTS_*_DIR y basenames.")

results_all = pd.concat(dfs, ignore_index=True)

# Nombre de modelo 'seguro' para emparejar con archivos OOS (que suelen venir sin espacios/paréntesis)
results_all["model_safe"] = results_all["model"].apply(sanitize_model_name)

# ------------------------------------------------------------
# Construimos métricas comparables (en tu caso: todo en NIVEL)
# ------------------------------------------------------------
# Selección explícita de columnas que vamos a usar para comparar
# (si alguna no existe, queda como NaN y no rompe el pipeline)
results_all["MAE_level"]  = pd.to_numeric(results_all["MAE_level"],  errors="coerce") if "MAE_level"  in results_all.columns else np.nan
results_all["RMSE_level"] = pd.to_numeric(results_all["RMSE_level"], errors="coerce") if "RMSE_level" in results_all.columns else np.nan
results_all["R2_level"]   = pd.to_numeric(results_all["R2_level"],   errors="coerce") if "R2_level"   in results_all.columns else np.nan

# Parametros para producción (JSON válido / None)
results_all["params_json"] = results_all["params"].apply(parse_params_to_json) if "params" in results_all.columns else None

# Orden para lectura
sort_cols = [c for c in ["h_tag", "MAE_level", "RMSE_level", "family", "model"] if c in results_all.columns]
results_all = results_all.sort_values(sort_cols).reset_index(drop=True)

display(results_all.head(12))
print("shape:", results_all.shape)


BENCH: c:\Users\jbrio\OneDrive - Universidad de Montevideo\Tesis MCD\Flujo de trabajo modelado y predicciones\Flujo sin BEVSA\results_n03\fx_results_benchmarks.csv
ML   : c:\Users\jbrio\OneDrive - Universidad de Montevideo\Tesis MCD\Flujo de trabajo modelado y predicciones\Flujo sin BEVSA\results_n04\fx_results_ml.csv
DL   : c:\Users\jbrio\OneDrive - Universidad de Montevideo\Tesis MCD\Flujo de trabajo modelado y predicciones\Flujo sin BEVSA\results_n05\fx_results_dl.csv


,h_tag,H_days,model,MAE_level,RMSE_level,R2_level,params,family,model_safe,params_json
0,12m,252,RandomForest,0.433686,0.550427,0.898775,"{'n_estimators': 300, 'min_samples_split': 2, ...",ml,RandomForest,"{""max_depth"": null, ""max_features"": ""sqrt"", ""m..."
1,12m,252,SVR_RBF,0.448687,0.573040,0.894537,"{'svr__C': np.float64(9.846738873614559), 'svr...",ml,SVR_RBF,None
2,12m,252,XGBoost,0.459613,0.903659,0.727167,"{'subsample': 1.0, 'reg_lambda': 0.0, 'reg_alp...",ml,XGBoost,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0..."
3,12m,252,LightGBM,0.471413,0.812310,0.779539,"{'subsample': 0.7, 'reg_lambda': 0.1, 'reg_alp...",ml,LightGBM,"{""colsample_bytree"": 0.6, ""learning_rate"": 0.0..."
4,12m,252,CNN_GRU,2.450054,3.086885,-2.704792,"{""units"": 64, ""dropout"": 0.3, ""lr"": 0.002}",dl,CNN_GRU,"{""dropout"": 0.3, ""lr"": 0.002, ""units"": 64}"
5,12m,252,ATTN_GRU,2.494349,3.034874,-2.580999,"{""units"": 128, ""dropout"": 0.3, ""lr"": 0.002}",dl,ATTN_GRU,"{""dropout"": 0.3, ""lr"": 0.002, ""units"": 128}"
6,12m,252,ATTN_LSTM,2.692689,3.340506,-3.338578,"{""units"": 96, ""dropout"": 0.3, ""lr"": 0.002}",dl,ATTN_LSTM,"{""dropout"": 0.3, ""lr"": 0.002, ""units"": 96}"
7,12m,252,CNN_LSTM,2.907796,3.574546,-3.967806,"{""units"": 128, ""dropout"": 0.1, ""lr"": 0.001}",dl,CNN_LSTM,"{""dropout"": 0.1, ""lr"": 0.001, ""units"": 128}"
8,12m,252,RandomWalk,3.020142,3.763407,0.532111,NaN,benchmark,RandomWalk,None
9,12m,252,RPPP,3.087984,3.504459,0.593661,NaN,benchmark,RPPP,None


shape: (55, 10)


## 3) Seleccionar mejor modelo por horizonte
Elegimos el modelo con menor `MAE_cmp` (en nivel si existe). Guardamos una tabla lista para producción.

In [4]:
# Reglas de desempate: MAE_level -> RMSE_level -> -R2_level -> family/model
tmp = results_all.copy()

tmp["model_safe"] = tmp["model"].apply(sanitize_model_name)

# Aseguramos numéricos (comparación)
tmp["MAE_level"]  = pd.to_numeric(tmp["MAE_level"], errors="coerce")
tmp["RMSE_level"] = pd.to_numeric(tmp["RMSE_level"], errors="coerce")
tmp["R2_level"]   = pd.to_numeric(tmp["R2_level"], errors="coerce")

tmp = tmp.dropna(subset=["MAE_level"]).copy()

tmp = tmp.sort_values(
    ["h_tag", "MAE_level", "RMSE_level", "R2_level", "family", "model_safe", "model"],
    ascending=[True, True, True, False, True, True, True]
)

best_models = tmp.groupby("h_tag", as_index=False).head(1).reset_index(drop=True)

# Campos recomendados para producción (selección explícita)
keep_cols = [
    "h_tag", "H_days", "family", "model", "model_safe",
    "MAE_level", "RMSE_level", "R2_level",
    "params_json", "params"
]
keep_cols = [c for c in keep_cols if c in best_models.columns]

best_models_out = best_models[keep_cols].copy()
best_models_out["metric_used"] = "MAE_level"

display(best_models_out)
print("best_models_per_horizon shape:", best_models_out.shape)

# Exportes
# (results_all incluye las métricas comparables y params_json)
results_all.to_csv(OUT_RESULTS_ALL_CSV, index=False)
results_all.to_excel(OUT_RESULTS_ALL_XLSX, index=False)

best_models_out.to_csv(OUT_BEST_CSV, index=False)
best_models_out.to_excel(OUT_BEST_XLSX, index=False)

print("✅ Guardado:")
print(" -", OUT_RESULTS_ALL_CSV)
print(" -", OUT_RESULTS_ALL_XLSX)
print(" -", OUT_BEST_CSV)
print(" -", OUT_BEST_XLSX)


,h_tag,H_days,family,model,model_safe,MAE_level,RMSE_level,R2_level,params_json,params,metric_used
0,12m,252,ml,RandomForest,RandomForest,0.433686,0.550427,0.898775,"{""max_depth"": null, ""max_features"": ""sqrt"", ""m...","{'n_estimators': 300, 'min_samples_split': 2, ...",MAE_level
1,1m,21,ml,RandomForest,RandomForest,0.364530,0.469680,0.920854,"{""max_depth"": 10, ""max_features"": ""log2"", ""min...","{'n_estimators': 500, 'min_samples_split': 2, ...",MAE_level
2,3m,63,ml,LightGBM,LightGBM,0.376786,0.489653,0.914790,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...","{'subsample': 0.85, 'reg_lambda': 0.0, 'reg_al...",MAE_level
3,6m,126,ml,LightGBM,LightGBM,0.475298,0.643978,0.855705,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...","{'subsample': 0.85, 'reg_lambda': 0.0, 'reg_al...",MAE_level


best_models_per_horizon shape: (4, 11)
✅ Guardado:
 - c:\Users\jbrio\OneDrive - Universidad de Montevideo\Tesis MCD\Flujo de trabajo modelado y predicciones\Flujo sin BEVSA\results_n06\fx_results_all.csv
 - c:\Users\jbrio\OneDrive - Universidad de Montevideo\Tesis MCD\Flujo de trabajo modelado y predicciones\Flujo sin BEVSA\results_n06\fx_results_all.xlsx
 - c:\Users\jbrio\OneDrive - Universidad de Montevideo\Tesis MCD\Flujo de trabajo modelado y predicciones\Flujo sin BEVSA\results_n06\best_models_per_horizon.csv
 - c:\Users\jbrio\OneDrive - Universidad de Montevideo\Tesis MCD\Flujo de trabajo modelado y predicciones\Flujo sin BEVSA\results_n06\best_models_per_horizon.xlsx


## 4) Cargar y unificar predicciones OOS en nivel
Leemos `oos_*.csv/xlsx` y generamos `all_oos_predictions` en **nivel**.

**Convención de salida**:
- `Fecha_origen`: fecha t
- `Fecha_objetivo`: fecha t+H (si existe)
- `Fecha_eval`: fecha usada para evaluar (prioriza `Fecha_objetivo`)
- `truth_level`: valor real en nivel
- `pred_level`: predicción en nivel

In [5]:
def parse_oos_filename(path: str):
    """Parsea oos_horizon_model.(csv|xlsx)"""
    base = os.path.basename(path)

    # Ignorar agregados/best
    if base.startswith("oos_best_"):
        return None

    # Formato requerido: oos_{horizon}_{model}.(csv|xlsx)
    m = re.match(r"^oos_([^_]+)_(.+)\.(csv|xlsx)$", base, flags=re.IGNORECASE)
    if not m:
        return None

    return {"h_tag": m.group(1), "model_safe": m.group(2), "kind": "oos"}

import numpy as np
import pandas as pd
import re

def _to_float_series(s: pd.Series) -> pd.Series:
    """
    Convierte una serie a float manejando:
    - 44,21
    - 44.21
    - 4.421,00
    - 4,421.00
    - strings con espacios
    """
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)

    s = s.astype(str).str.strip()

    def parse_number(x):
        if x == "" or x.lower() in ["nan", "none"]:
            return np.nan

        # Detectar si tiene ambos separadores
        if "," in x and "." in x:
            # Caso europeo: 4.421,00
            if x.rfind(",") > x.rfind("."):
                x = x.replace(".", "").replace(",", ".")
            # Caso US: 4,421.00
            else:
                x = x.replace(",", "")
        else:
            # Solo coma → decimal europeo
            if "," in x:
                x = x.replace(",", ".")
            # Solo punto → ya es decimal correcto

        try:
            return float(x)
        except:
            return np.nan

    return s.apply(parse_number)

def read_oos_smart(path: str) -> pd.DataFrame:
    """Lee OOS con separador desconocido y posible coma decimal."""
    ext = os.path.splitext(path)[1].lower()

    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    if ext == ".csv":
        # Intento 1: inferir separador (engine python)
        try:
            return pd.read_csv(path, sep=None, engine="python")
        except Exception:
            pass
        # Intento 2: separador ';' (común cuando hay coma decimal)
        try:
            return pd.read_csv(path, sep=";")
        except Exception:
            pass
        # Intento 3: separador '\t' (tab)
        return pd.read_csv(path, sep="\t")

    raise ValueError(f"Extensión no soportada: {ext} | {path}")

def standardize_oos_df(df: pd.DataFrame, meta: dict, path: str) -> pd.DataFrame | None:
    """Devuelve DF estándar con columnas:
    h_tag, model_safe, fecha, fecha_objetivo, dolar_t, pred_level, truth_level, Fecha_eval
    """
    # Trabajamos case-insensitive
    lower_map = {c: c.lower() for c in df.columns}
    df2 = df.rename(columns=lower_map).copy()
    cols = set(df2.columns)

    needed = {"fecha", "fecha_objetivo", "dolar_t", "dolar_pred_th", "dolar_real_th"}
    if not needed.issubset(cols):
        return None

    out = pd.DataFrame()

    # Selección explícita de columnas necesarias
    out["fecha"]          = pd.to_datetime(df2["fecha"], errors="coerce")
    out["fecha_objetivo"] = pd.to_datetime(df2["fecha_objetivo"], errors="coerce")

    out["dolar_t"]        = _to_float_series(df2["dolar_t"])
    out["pred_level"]     = _to_float_series(df2["dolar_pred_th"])
    out["truth_level"]    = _to_float_series(df2["dolar_real_th"])

    # Para evaluación OOS usamos la fecha objetivo (lo que realmente queremos predecir)
    out["Fecha_eval"] = out["fecha_objetivo"]

    out["h_tag"] = meta["h_tag"]
    out["model_safe"] = meta["model_safe"]

    # Limpieza mínima
    out = out.dropna(subset=["Fecha_eval", "pred_level", "truth_level"])
    return out

# ------------------------------------------------------------
# Recolectar archivos OOS desde las carpetas definidas
# ------------------------------------------------------------
oos_files = []
for d in OOS_DIRS:
    if os.path.isdir(d):
        oos_files.extend(glob.glob(os.path.join(d, "oos_*.csv")))
        oos_files.extend(glob.glob(os.path.join(d, "oos_*.xlsx")))

print("OOS files encontrados:", len(oos_files))

rows = []
skipped = []
for path in sorted(set(oos_files)):
    meta = parse_oos_filename(path)
    if meta is None:
        continue
    try:
        df = read_oos_smart(path)
        std = standardize_oos_df(df, meta, path)
        if std is None or std.empty:
            skipped.append(path)
            continue
        rows.append(std)
    except Exception:
        skipped.append(path)
        continue

if not rows:
    print("⚠️ No pude construir all_oos_predictions (no encontré OOS válidos). Se omiten secciones 4-6.")
    all_oos = pd.DataFrame(columns=["h_tag","model_safe","fecha","fecha_objetivo","dolar_t","pred_level","truth_level","Fecha_eval"])
else:
    all_oos = pd.concat(rows, ignore_index=True)

# Orden y sanity
all_oos = all_oos.sort_values(["h_tag", "model_safe", "Fecha_eval"]).reset_index(drop=True)

# Selección explícita final de columnas que dejamos en el consolidado
ALL_OOS_COLS = ["h_tag", "model_safe", "fecha", "fecha_objetivo", "dolar_t", "pred_level", "truth_level", "Fecha_eval"]
all_oos = all_oos[[c for c in ALL_OOS_COLS if c in all_oos.columns]].copy()

display(all_oos.head(10))
print("shape:", all_oos.shape)
print("skipped:", len(skipped))
if skipped:
    print("Ejemplos skipped:")
    for s in skipped[:10]:
        print(" -", s)


OOS files encontrados: 126


,h_tag,model_safe,fecha,fecha_objetivo,dolar_t,pred_level,truth_level,Fecha_eval
0,12m,"ARIMA1,0,0",2020-10-26,2021-11-01,42.65,46.232394,44.15,2021-11-01
1,12m,"ARIMA1,0,0",2020-10-26,2021-11-01,42.65,46.232394,44.15,2021-11-01
2,12m,"ARIMA1,0,0",2020-10-27,2021-11-03,42.73,46.317626,44.13,2021-11-03
3,12m,"ARIMA1,0,0",2020-10-27,2021-11-03,42.73,46.317626,44.13,2021-11-03
4,12m,"ARIMA1,0,0",2020-10-28,2021-11-04,42.91,46.511172,43.90,2021-11-04
5,12m,"ARIMA1,0,0",2020-10-28,2021-11-04,42.91,46.511172,43.90,2021-11-04
6,12m,"ARIMA1,0,0",2020-10-29,2021-11-05,42.95,46.552657,43.62,2021-11-05
7,12m,"ARIMA1,0,0",2020-10-29,2021-11-05,42.95,46.552657,43.62,2021-11-05
8,12m,"ARIMA1,0,0",2020-10-30,2021-11-08,43.04,46.648095,43.51,2021-11-08
9,12m,"ARIMA1,0,0",2020-10-30,2021-11-08,43.04,46.648095,43.51,2021-11-08


shape: (107448, 8)
skipped: 0


## 5) Chequeos de calidad y exportes OOS
Validamos duplicados por fecha de evaluación y exportamos `all_oos_predictions`.

In [6]:

# Duplicados por h_tag/model/Fecha_eval (si existen, los reportamos)
dup = (all_oos
       .groupby(["h_tag", "model_safe", "Fecha_eval"])
       .size()
       .reset_index(name="n")
       .query("n > 1")
)
print("Duplicados (h_tag/model/Fecha_eval) >", 0, ":", len(dup))
if len(dup) > 0:
    display(dup.head(20))

# Exportes
all_oos.to_csv(OUT_OOS_ALL_CSV, index=False)
all_oos.to_excel(OUT_OOS_ALL_XLSX, index=False)

print("✅ Guardado:")
print(" -", OUT_OOS_ALL_CSV)
print(" -", OUT_OOS_ALL_XLSX)


Duplicados (h_tag/model/Fecha_eval) > 0 : 53724


,h_tag,model_safe,Fecha_eval,n
0,12m,"ARIMA1,0,0",2021-11-01,2
1,12m,"ARIMA1,0,0",2021-11-03,2
2,12m,"ARIMA1,0,0",2021-11-04,2
3,12m,"ARIMA1,0,0",2021-11-05,2
4,12m,"ARIMA1,0,0",2021-11-08,2
5,12m,"ARIMA1,0,0",2021-11-09,2
6,12m,"ARIMA1,0,0",2021-11-10,2
7,12m,"ARIMA1,0,0",2021-11-11,2
8,12m,"ARIMA1,0,0",2021-11-12,2
9,12m,"ARIMA1,0,0",2021-11-15,2


✅ Guardado:
 - c:\Users\jbrio\OneDrive - Universidad de Montevideo\Tesis MCD\Flujo de trabajo modelado y predicciones\Flujo sin BEVSA\results_n06\all_oos_predictions.csv
 - c:\Users\jbrio\OneDrive - Universidad de Montevideo\Tesis MCD\Flujo de trabajo modelado y predicciones\Flujo sin BEVSA\results_n06\all_oos_predictions.xlsx


## 6) (Opcional) OOS solo para los mejores modelos
Genera un archivo con OOS únicamente para el modelo seleccionado en cada horizonte (útil para auditoría y gráficos).

In [7]:
best_key = best_models_out[["h_tag", "model_safe"]].drop_duplicates()
best_oos = all_oos.merge(best_key, on=["h_tag", "model_safe"], how="inner").copy()
best_oos = best_oos.sort_values(["h_tag", "Fecha_eval"]).reset_index(drop=True)

# ------------------------------------------------------------
# Export en el MISMO formato que tus archivos OOS
# ------------------------------------------------------------
best_oos_export = best_oos.copy()
best_oos_export["dolar_pred_tH"] = best_oos_export["pred_level"]
best_oos_export["dolar_real_tH"] = best_oos_export["truth_level"]

OOS_EXPORT_COLS = ["h_tag", "model_safe", "fecha", "fecha_objetivo", "dolar_t", "dolar_pred_tH", "dolar_real_tH"]
best_oos_export = best_oos_export[[c for c in OOS_EXPORT_COLS if c in best_oos_export.columns]].copy()


# Además, exportamos un archivo por horizonte (útil para producción)
for h in sorted(best_oos_export["h_tag"].dropna().unique()):
    tmp_h = best_oos_export.query("h_tag == @h").copy()
    # Para estos archivos, dejamos solo columnas estándar del OOS (más fácil de consumir)
    tmp_h_std = tmp_h[["fecha", "fecha_objetivo", "dolar_t", "dolar_pred_tH", "dolar_real_tH"]].copy()
    tmp_h_std = tmp_h_std.sort_values("fecha").reset_index(drop=True)

    out_csv  = os.path.join(OUT_DIR, f"oos_best_{h}.csv")
    out_xlsx = os.path.join(OUT_DIR, f"oos_best_{h}.xlsx")
    tmp_h_std.to_csv(out_csv, index=False)
    tmp_h_std.to_excel(out_xlsx, index=False)


display(best_oos_export.head(10))
print("best_oos_export shape:", best_oos_export.shape)

OUT_BEST_OOS_CSV  = os.path.join(OUT_DIR, "oos_best_models.csv")
OUT_BEST_OOS_XLSX = os.path.join(OUT_DIR, "oos_best_models.xlsx")
best_oos_export.to_csv(OUT_BEST_OOS_CSV, index=False)
best_oos_export.to_excel(OUT_BEST_OOS_XLSX, index=False)

print("✅ Guardado:")
print(" -", OUT_BEST_OOS_CSV)
print(" -", OUT_BEST_OOS_XLSX)


,h_tag,model_safe,fecha,fecha_objetivo,dolar_t,dolar_pred_tH,dolar_real_tH
0,12m,RandomForest,2022-01-03,2023-01-05,44.61,42.049901,39.90
1,12m,RandomForest,2022-01-03,2023-01-05,44.61,42.049901,39.90
2,12m,RandomForest,2022-01-04,2023-01-09,44.73,41.775010,39.85
3,12m,RandomForest,2022-01-04,2023-01-09,44.73,41.775010,39.85
4,12m,RandomForest,2022-01-05,2023-01-10,44.73,41.217461,39.78
5,12m,RandomForest,2022-01-05,2023-01-10,44.73,41.217461,39.78
6,12m,RandomForest,2022-01-07,2023-01-11,44.56,40.597753,39.80
7,12m,RandomForest,2022-01-07,2023-01-11,44.56,40.597753,39.80
8,12m,RandomForest,2022-01-10,2023-01-12,44.70,40.615974,39.67
9,12m,RandomForest,2022-01-10,2023-01-12,44.70,40.615974,39.67


best_oos_export shape: (5626, 7)
✅ Guardado:
 - c:\Users\jbrio\OneDrive - Universidad de Montevideo\Tesis MCD\Flujo de trabajo modelado y predicciones\Flujo sin BEVSA\results_n06\oos_best_models.csv
 - c:\Users\jbrio\OneDrive - Universidad de Montevideo\Tesis MCD\Flujo de trabajo modelado y predicciones\Flujo sin BEVSA\results_n06\oos_best_models.xlsx
